## Libraries

In [29]:
import requests
import pandas as pd
from tqdm import tqdm
import networkx as nx

from matplotlib import pyplot as plt
from time import time

from pyvis.network import Network

### Helper functions

In [30]:
# Helper functions

def pyvis_show(graf, phi=False, name="Default"):
    net = Network()
    net.from_nx(graf)
    net.toggle_physics(phi)
    net.show(f"{name}_{time()}.html", notebook=False)

def networkx_plot(graf, name="Default"):
    node_sizes = [50*len(graf.edges(n)) for n in graf.nodes()]
    edges = graf.edges()
    #weights = [graf[u][v]['weight']/100 for u,v in edges]
    nx.draw_networkx(graf, pos=nx.kamada_kawai_layout(graf), node_size=node_sizes)
    plt.show() 

## Data gathering from Croris API

In [16]:
# Step 1: Get the list of publications for the institution
institution_url = "https://www.croris.hr/crosbi-api/ustanova/289"

# Fetch the list of publications
response = requests.get(institution_url)
if response.status_code == 200:
    data = response.json()
    publications = data['_links']['publikacije']  # Get the list of publications
else:
    print(f"Error fetching data from {institution_url}. Status code: {response.status_code}")

def authorstringlist2dict(stringlist):
    """
    Converts a string of authors (formatted as 'surname, name', separated by semicolons) 
    into a list of dictionaries with 'name', 'surname', 'authorId', and 'role'.

    Args:
        stringlist (str): Authors in the format 'surname, name', separated by ';'.

    Returns:
        list: List of dictionaries, each containing author details.
    """
    authorlist = []
    if ";" in stringlist:
        for author in stringlist.split(";"):
            authorlist.append({"name": author.split(",")[1], "surname": author.split(",")[0], "authorId": author.replace(",", "").replace(" ", ""), "role": {'id': 905, 'naziv': 'autor/i'}})
    else:
        for author in [stringlist]:
            try:
                authorlist.append({"name": author.split(",")[1], "surname": author.split(",")[0], "authorId": author.replace(",", "").replace(" ", ""), "role": {'id': 905, 'naziv': 'autor/i'}})
            except IndexError:
                authorlist.append({"name": author.split(" ")[1], "surname": author.split(" ")[0], "authorId": author.replace(",", "").replace(" ", ""), "role": {'id': 905, 'naziv': 'autor/i'}})
        
    return authorlist

data_list = []
# Step 2: Loop through each publication and fetch details
for publication in tqdm(publications):
    publication_url = publication['href']  # URL for each publication
    pub_response = requests.get(publication_url)
    
    # print(pub_response.json())
    
    if pub_response.status_code == 200:
        pub_data = pub_response.json()
        # Process each publication (e.g., get the authors)
        # print(f"Title: {pub_data.get('naslov')}")
        title = pub_data.get('naslov')
        authors = []
        if 'osobeResources' in pub_data:
            superviser_flag = False
            for author in pub_data['osobeResources']['_embedded']['osobe']:
                if author['funkcija']['id'] in [905, 907]:
                    if author['funkcija']['id'] == 907:
                        superviser_flag = True
                    # print(f" - {author['ime']} {author['prezime']} {author['crorisId']}")
                    authors.append({"name": author['ime'], "surname": author['prezime'], "authorId": author['crorisId'], "role": author['funkcija']})
                    
        else:
            print("No authors found")
        
        # Add authors with no ID in the database (students)
        if superviser_flag:
            authors.extend(authorstringlist2dict(pub_data.get('autori')))
        
        if len(authors) < 1:
            authors.append("no_authors")
            
        year = pub_data.get('godina')
        DOI = pub_data.get('crosbiId')
        
        paper_data = {
            "Title": title,
            "DOI": DOI,
            "Authors": authors
        }
        
        data_list.append(paper_data)
        
    else:
        print(f"Error fetching data from {publication_url}. Status code: {pub_response.status_code}")
        
df = pd.DataFrame(data_list)

 21%|██▏       | 222/1038 [00:24<01:29,  9.13it/s]

No authors found


 42%|████▏     | 434/1038 [00:48<01:08,  8.87it/s]

No authors found


 46%|████▋     | 482/1038 [00:53<01:01,  9.04it/s]

...

No authors found


 99%|█████████▊| 1023/1038 [02:01<00:01,  9.11it/s]

No authors found
No authors found


100%|██████████| 1038/1038 [02:02<00:00,  8.45it/s]

No authors found


### Data info

In [24]:
print(df.info())
# Calculate the percentage of rows where the "Authors" column contains the list ["no_authors"]
percentage = len(df[df["Authors"].apply(lambda x: x == ["no_authors"])]) / len(df) * 100

# Print the percentage
print(f"\nPercentage of papers with no aparent authors: {percentage:.2f} %\n")
print(df[df["Authors"].apply(lambda x: x == ["no_authors"])])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1038 entries, 0 to 1037
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Title    1038 non-null   object
 1   DOI      1038 non-null   int64 
 2   Authors  1038 non-null   object
dtypes: int64(1), object(2)
memory usage: 24.5+ KB
None

Percentage of papers with no aparent authors: 1.54 %

                                                  Title     DOI       Authors
0     2. Radionica znanstvenog programa "Napredne mr...    7710  [no_authors]
1     3. Radionica znanstvenog programa "Napredne mr...    7711  [no_authors]
2                                   Speech Technologies    8806  [no_authors]
3                      Speech and Language Technologies    8809  [no_authors]
4     4. Radionica znanstvenog programa "Napredne mr...   10385  [no_authors]
15    Znanstveni skup doktorskih studenata informati...   23191  [no_authors]
480           Automobile liability insurance data mo

### Data export

In [25]:
df.to_csv(f"./DATA/crosbi_data_{len(df)}.csv")
df.to_pickle(f"./DATA/crosbi_data_{len(df)}.pickle")

## Data loading

In [26]:
DIR = "./DATA/crosbi_data_1038.pickle"

df = pd.read_pickle(DIR)

In [59]:
# Create a graph
G = nx.Graph()
N = 1200
for k, item in enumerate(df.iterrows()):
    if item[1]["Authors"][0] == "no_authors":
        continue
    
    print(item[1]["Authors"][0]['authorId'])
    

    authors = [(author['authorId'], author['name'] + " " + author['surname']) for author in item[1]["Authors"]]
    print(authors)
    
    for i, author1 in enumerate(authors):
        for j, author2 in enumerate(authors):
            if i != j:
                G.add_node(author1[0], label=author1[1])
                G.add_node(author2[0], label=author2[1])
                G.add_edge(author1[0], author2[0])       
    if k > N:
        break
# Plot the graph
pyvis_show(G, phi=True, name="author_reference_graph")

8822
[(8822, 'Mile Pavlić')]
8822
[(8822, 'Mile Pavlić'), (22062, 'Velimir Srića')]
22062
[(22062, 'Velimir Srića'), (8822, 'Mile Pavlić')]
8822
[(8822, 'Mile Pavlić')]
8822
[(8822, 'Mile Pavlić')]
22498
[(22498, 'Mario Radovan')]
22498
[(22498, 'Mario Radovan')]
22498
[(22498, 'Mario Radovan')]
2053
[(2053, 'Sanda Martinčić-Ipšić'), (2048, 'Ana Meštrović')]
7707
[(7707, 'Nataša Hoić-Božić'), (28273, 'Martina Holenko Dlab')]
7707
[(7707, 'Nataša Hoić-Božić'), (14252, 'Vedran Mornar'), (27160, 'Ivica Botički')]
6149
[(6149, 'Božidar Kovačić'), (11489, 'Maja Matetić'), (4644, 'Marija Brkić Bakarić')]
2053
[(2053, 'Sanda Martinčić-Ipšić'), (5033, 'Ivo Ipšić')]
4644
[(4644, 'Marija Brkić Bakarić'), (5564, 'Sanja Seljan'), (11489, 'Maja Matetić')]
8439
[(8439, 'Vlasta Kučiš'), (4644, 'Marija Brkić Bakarić'), (5564, 'Sanja Seljan')]
15976
[(15976, 'Damir Šimunović')]
2053
[(2053, 'Sanda Martinčić-Ipšić'), (29203, 'Lucia Načinović Prskalo')]
2053
[(2053, 'Sanda Martinčić-Ipšić'), (2048, 'Ana 